In [ ]:
import astropy.io.fits as pf
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np
import matplotlib.pyplot as plt
import math
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LogNorm
import os
import subprocess
import importlib as imp
import copy
import scipy.stats
from matplotlib.pyplot import cm
from scipy.stats import kde
import figure_subroutines as subs
from astropy import units as u
from astropy.coordinates import SkyCoord
from pylab import *
from matplotlib import ticker

In [ ]:
step_files = ['THESIS_pics/01_tapered_gmims.fits','THESIS_pics/02_initial_tapered_gmims.fits',
              'THESIS_pics/03_deconvolved_gmims.fits', 'THESIS_pics/04_lowpass_filtered_gmims.fits',
              'THESIS_pics/05_deconvolved_gmims_image.fits','THESIS_pics/06_gmims_primary_beam_match.fits',
              'THESIS_pics/07_gmims_stprimary.fits','THESIS_pics/08_feathered_gmims.fits',
              'THESIS_pics/09_final_gmims.fits','THESIS_pics/10_initial_st.fits',
              'THESIS_pics/11_initial_st_uv.fits','THESIS_pics/12_feathered_st.fits',
              'THESIS_pics/13_final_st.fits']


image_plane = []

for i in [0,4,5,8,9,12]:
    field = fits.open(step_files[i])
    hdr1 = field[0].header
    data = field[0].data[0,0,:,:]
    image_plane.append(data)
    print(data.shape)
hdr1['NAXIS'] = 2
del hdr1['NAXIS3']
del hdr1['NAXIS4']
del hdr1['CTYPE3']                   
del hdr1['CRVAL3']                    
del hdr1['CRPIX3']                       
del hdr1['CDELT3']             
del hdr1['CROTA3']                     
del hdr1['CTYPE4']              
del hdr1['CRVAL4']  
del hdr1['CRPIX4']                      
del hdr1['CDELT4']                   
del hdr1['CROTA4']
#print(repr(hdr))
wcs = WCS(hdr1)
print(wcs)
print('')

uv_plane = []

for i in [1,2,3,6,7,10,11]:
    field = fits.open(step_files[i])
    hdr2 = field[0].header
    data = field[0].data[0,0,:,:]
    uv_plane.append(data)
    print(data.shape)

uvdelt = hdr2['CDELT2']
print(uvdelt)

In [ ]:
fs = 22
bottom = 0.07
top = 0.98
right = 0.96
left = 0.04
PImin = -0.1
PImax = 0.1
cbticks = [-0.1,-0.05,0.,0.05,0.1]
cbticksuv = [0,1,2,3]
numpix = 41
shrink = 1.

axs = ['ax1','ax2','ax3','ax4','ax5']

ext_step = ((numpix-1)/2)*uvdelt+uvdelt/2
extent2=[-ext_step,ext_step,-ext_step,ext_step]

fig = plt.figure(figsize=(20,15))

plt.subplots_adjust(wspace=0.2,hspace=0.3)
plt.subplots_adjust(top = top, bottom = bottom, right = right, left = left)

axs[0] = fig.add_subplot(3,3,1,projection=wcs)
image1 = axs[0].imshow(image_plane[0],origin='lower',vmin=PImin,vmax=PImax,cmap='gray')
cb = fig.colorbar(image1, orientation='vertical',shrink=shrink,ax=axs[0],ticks=cbticks)
cb.set_label(label='K',size=fs)
cb.ax.tick_params(labelsize=fs)


axs[1] = fig.add_subplot(3,3,4)
ampl = uv_plane[0][1:1024,0:513]
ampl_refl = np.flip(np.flip(ampl,axis=0),axis=1)
ampl_new = np.concatenate([ampl_refl[:,0:512],ampl],axis=1)[:,1:1024]
image2 = axs[1].imshow(ampl_new[511-(numpix-1)/2:511+(numpix+1)/2,511-(numpix-1)/2:511+(numpix+1)/2]/1000,
           vmin=0,vmax=3,origin='bottom',cmap='gray',extent=extent2)
cb = fig.colorbar(image2, orientation='vertical',shrink=shrink,ax=axs[1],ticks=cbticksuv)
cb.set_label(label='$\\times$ 10 K - uv-plane equiv.',size=fs)
cb.ax.tick_params(labelsize=fs)



axs[2] = fig.add_subplot(3,3,5)
ampl = uv_plane[1][1:1024,0:513]
ampl_refl = np.flip(np.flip(ampl,axis=0),axis=1)
ampl_new = np.concatenate([ampl_refl[:,0:512],ampl],axis=1)[:,1:1024]
image3 = axs[2].imshow(ampl_new[511-(numpix-1)/2:511+(numpix+1)/2,511-(numpix-1)/2:511+(numpix+1)/2]/1000,
           vmin=0,vmax=3,origin='bottom',cmap='gray',extent=extent2)
cb = fig.colorbar(image3, orientation='vertical',shrink=shrink,ax=axs[2],ticks=cbticksuv)
cb.set_label(label='$\\times$ 10 K - uv-plane equiv.',size=fs)
cb.ax.tick_params(labelsize=fs)

axs[3] = fig.add_subplot(3,3,6)
ampl = uv_plane[2][1:1024,0:513]
ampl_refl = np.flip(np.flip(ampl,axis=0),axis=1)
ampl_new = np.concatenate([ampl_refl[:,0:512],ampl],axis=1)[:,1:1024]
image4 = axs[3].imshow(ampl_new[511-(numpix-1)/2:511+(numpix+1)/2,511-(numpix-1)/2:511+(numpix+1)/2]/1000,
           vmin=0,vmax=3,origin='bottom',cmap='gray',extent=extent2)
cb = fig.colorbar(image4, orientation='vertical',shrink=shrink,ax=axs[3],ticks=cbticksuv)
cb.set_label(label='$\\times$ 10 K - uv-plane equiv.',size=fs)
cb.ax.tick_params(labelsize=fs)


axs[4] = fig.add_subplot(3,3,9,projection=wcs)
image5 = axs[4].imshow(image_plane[1],origin='lower',vmin=PImin,vmax=PImax,cmap='gray')
cb = fig.colorbar(image5, orientation='vertical',shrink=shrink,ax=axs[4],ticks=cbticks)
cb.set_label(label='K',size=fs)
cb.ax.tick_params(labelsize=fs)


for i in [1,2,3]:
    axs[i].tick_params(axis="x", labelsize=fs)
    axs[i].tick_params(axis="y", labelsize=fs)
    axs[i].set_xlabel('u (m)',fontsize=fs)
    axs[i].set_ylabel('v (m)',fontsize=fs,labelpad=-5)
    axs[i].set_xticks([-40,-20,0,20,40])
    axs[i].set_yticks([-40,-20,0,20,40])

for i in [0,4]:
    axs[i].coords[0].set_major_formatter('hh:mm')
    axs[i].coords[0].set_ticks([300.,297.5,295.5]*u.degree)
    axs[i].coords[1].set_major_formatter('dd')
    axs[i].coords[0].set_ticklabel(size=fs)
    axs[i].coords[1].set_ticklabel(size=fs)
    axs[i].coords[0].set_axislabel('Right Ascension (J2000)', fontsize=fs)
    axs[i].coords[1].set_axislabel('Declination (J2000)', fontsize=fs)

x1 = 0.16
x2 = 0.4855
x3 = 0.81
y1 = 0.3
y2 = 0.33
y3 = 0.7
y4 = 0.67
dax = 0.01
day = 0.015
lw = 2.5
    
plt.plot([x1,x2], [y1,y1], lw=lw,transform=gcf().transFigure, clip_on=False,color='black')
plt.plot([x2,x3], [y3,y3], lw=lw,transform=gcf().transFigure, clip_on=False,color='black')

plt.plot([x1,x1], [y1,y2], lw=lw,transform=gcf().transFigure, clip_on=False,color='black')
plt.plot([x2,x2], [y1,y2], lw=lw,transform=gcf().transFigure, clip_on=False,color='black')

plt.plot([x2,x2], [y3,y4], lw=lw,transform=gcf().transFigure, clip_on=False,color='black')
plt.plot([x3,x3], [y3,y4], lw=lw,transform=gcf().transFigure, clip_on=False,color='black')

plt.plot([x2,x2+dax], [y2,y2-day], lw=lw,transform=gcf().transFigure, clip_on=False,color='black')
plt.plot([x2,x2-dax], [y2,y2-day], lw=lw,transform=gcf().transFigure, clip_on=False,color='black')

plt.plot([x3,x3+dax], [y4,y4+day], lw=lw,transform=gcf().transFigure, clip_on=False,color='black')
plt.plot([x3,x3-dax], [y4,y4+day], lw=lw,transform=gcf().transFigure, clip_on=False,color='black')

plt.text(0.21,0.27,'Divide by SA beam transform',transform=gcf().transFigure, clip_on=False,fontsize=fs)
plt.text(0.6,0.72,'Low pass filter',transform=gcf().transFigure, clip_on=False,fontsize=fs)

plt.savefig('FIGURES/combining_steps_first.pdf')

In [ ]:
fs = 20
bottom = 0.05
top = 0.98
right = 0.96
left = 0.05
PImin = -0.1
PImax = 0.1
numpix = 81
shrink = 1.
cbticks = [-0.1,-0.05,0.,0.05,0.1]
cbticksuv1 = [0.0,0.5,1.0,1.5,2.0]
cbticksuv2 = [0.0,0.1,0.2,0.3,0.4,0.5]

fig = plt.figure(figsize=(13.5,20))
axs = ['ax1','ax2','ax3','ax4','ax5','ax6','ax7','ax8']

ext_step = ((numpix-1)/2)*uvdelt+uvdelt/2
extent2=[-ext_step,ext_step,-ext_step,ext_step]

plt.subplots_adjust(wspace=0.3,hspace=0.3)
plt.subplots_adjust(top = top, bottom = bottom, right = right, left = left)


# SA image matched to ST primary beam:
axs[0] = fig.add_subplot(4,2,1,projection=wcs)
image1 = axs[0].imshow(image_plane[2],origin='lower',vmin=PImin,vmax=PImax,cmap='gray')
cb = fig.colorbar(image1, orientation='vertical',shrink=shrink,ticks=cbticks)
cb.set_label(label='K',size=fs,labelpad=-5)
cb.ax.tick_params(labelsize=fs)

# ST image initial:
axs[1] = fig.add_subplot(4,2,2,projection=wcs)
image2 = axs[1].imshow(image_plane[4],origin='lower',vmin=PImin,vmax=PImax,cmap='gray')
cb = fig.colorbar(image2, orientation='vertical',shrink=shrink,ticks=cbticks)
cb.set_label(label='K',size=fs,labelpad=-5)
cb.ax.tick_params(labelsize=fs)


# SA in uv matched to ST primary beam:
axs[2] = fig.add_subplot(4,2,3)
ampl = uv_plane[3][1:1024,0:513]
ampl_refl = np.flip(np.flip(ampl,axis=0),axis=1)
ampl_new = np.concatenate([ampl_refl[:,0:512],ampl],axis=1)[:,1:1024]
image3 = axs[2].imshow(ampl_new[511-(numpix-1)/2:511+(numpix+1)/2,511-(numpix-1)/2:511+(numpix+1)/2]/1000,
           vmin=0,vmax=2,origin='bottom',cmap='gray',extent=extent2)
cb = fig.colorbar(image3, orientation='vertical',shrink=shrink,ticks=cbticksuv1)
cb.set_label(label='$\\times$ 10 K - uv-plane equiv.',size=fs)
cb.ax.tick_params(labelsize=fs)

# ST in uv original:
axs[3] = fig.add_subplot(4,2,4)
ampl = uv_plane[5][1:1024,0:513]
ampl_refl = np.flip(np.flip(ampl,axis=0),axis=1)
ampl_new = np.concatenate([ampl_refl[:,0:512],ampl],axis=1)[:,1:1024]
image4 = axs[3].imshow(ampl_new[511-(numpix-1)/2:511+(numpix+1)/2,511-(numpix-1)/2:511+(numpix+1)/2]/1000,
           vmin=0,vmax=0.5,origin='bottom',cmap='gray',extent=extent2)
cb = fig.colorbar(image4, orientation='vertical',shrink=shrink,ticks=cbticksuv2)
cb.set_label(label='$\\times$ 10 K - uv-plane equiv.',size=fs)
cb.ax.tick_params(labelsize=fs)


# SA in uv feathered:
axs[4] = fig.add_subplot(4,2,5)
ampl = uv_plane[4][1:1024,0:513]
ampl_refl = np.flip(np.flip(ampl,axis=0),axis=1)
ampl_new = np.concatenate([ampl_refl[:,0:512],ampl],axis=1)[:,1:1024]
image5 = axs[4].imshow(ampl_new[511-(numpix-1)/2:511+(numpix+1)/2,511-(numpix-1)/2:511+(numpix+1)/2]/1000,
           vmin=0,vmax=2,origin='bottom',cmap='gray',extent=extent2)
cb = fig.colorbar(image5, orientation='vertical',shrink=shrink,ticks=cbticksuv1)
cb.set_label(label='$\\times$ 10 K - uv-plane equiv.',size=fs)
cb.ax.tick_params(labelsize=fs)

# ST in uv feathered:
axs[5] = fig.add_subplot(4,2,6)
ampl = uv_plane[6][1:1024,0:513]
ampl_refl = np.flip(np.flip(ampl,axis=0),axis=1)
ampl_new = np.concatenate([ampl_refl[:,0:512],ampl],axis=1)[:,1:1024]
image6 = axs[5].imshow(ampl_new[511-(numpix-1)/2:511+(numpix+1)/2,511-(numpix-1)/2:511+(numpix+1)/2]/1000,
           vmin=0,vmax=0.5,origin='bottom',cmap='gray',extent=extent2)
cb = fig.colorbar(image6, orientation='vertical',shrink=shrink,ticks=cbticksuv2)
cb.set_label(label='$\\times$ 10 K - uv-plane equiv.',size=fs)
cb.ax.tick_params(labelsize=fs)


# SA final image:
axs[6] = fig.add_subplot(4,2,7,projection=wcs)
image7 = axs[6].imshow(image_plane[3],origin='lower',vmin=PImin,vmax=PImax,cmap='gray')
cb = fig.colorbar(image7, orientation='vertical',shrink=shrink,ticks=cbticks)
cb.set_label(label='K',size=fs,labelpad=-5)
cb.ax.tick_params(labelsize=fs)

# ST final image:
axs[7] = fig.add_subplot(4,2,8,projection=wcs)
image8 = axs[7].imshow(image_plane[5],origin='lower',vmin=PImin,vmax=PImax,cmap='gray')
cb = fig.colorbar(image8, orientation='vertical',shrink=shrink,ticks=cbticks)
cb.set_label(label='K',size=fs,labelpad=-5)
cb.ax.tick_params(labelsize=fs)


for i in [2,3,4,5]:
    axs[i].tick_params(axis="x", labelsize=fs)
    axs[i].tick_params(axis="y", labelsize=fs)
    axs[i].set_xlabel('u (m)',fontsize=fs)
    axs[i].set_ylabel('v (m)',fontsize=fs,labelpad=-5)

for i in [0,1,6,7]:
    axs[i].coords[0].set_major_formatter('hh:mm')
    axs[i].coords[0].set_ticks([300.,297.5,295.5]*u.degree)
    axs[i].coords[1].set_major_formatter('dd')
    axs[i].coords[0].set_ticks_position('br')
    axs[i].coords[0].set_ticklabel(size=fs)
    axs[i].coords[1].set_ticklabel(size=fs)
    axs[i].coords[0].set_axislabel('Right Ascension (J2000)', fontsize=fs)
    axs[i].coords[1].set_axislabel('Declination (J2000)', fontsize=fs)
    
plt.savefig('FIGURES/combining_steps_second.pdf')

In [ ]:
#=============== SA initial ======================================
ampl_G_initial = uv_plane[0][1:1024,0:513]
print(ampl_G_initial[0,0])
print(ampl_G_initial[511,0])
print(ampl_G_initial[1022,0])
print(ampl_G_initial.shape)
u,v = subs.make_axis_lists_2D(hdr2)
u_G = np.flip(u[1:514],axis=0)
v_G = v[1:1024]
print(np.min(u_G), np.max(u_G), u_G.shape)
print(np.min(v_G), np.max(v_G), v_G.shape)
print('')

#=============== SA matched ======================================
ampl_G_match = uv_plane[3][1:1024,0:513]
print(ampl_G_match[0,0])
print(ampl_G_match[511,0])
print(ampl_G_match[1022,0])
print(ampl_G_match.shape)
u,v = subs.make_axis_lists_2D(hdr2)
u_G = np.flip(u[1:514],axis=0)
v_G = v[1:1024]
print(np.min(u_G), np.max(u_G), u_G.shape)
print(np.min(v_G), np.max(v_G), v_G.shape)
print('')

#=============== SA feathered ======================================
ampl_G_feath = uv_plane[4][1:1024,0:513]
print(ampl_G_feath[0,0])
print(ampl_G_feath[511,0])
print(ampl_G_feath[1022,0])
print(ampl_G_feath.shape)
u,v = subs.make_axis_lists_2D(hdr2)
u_G = np.flip(u[1:514],axis=0)
v_G = v[1:1024]
print(np.min(u_G), np.max(u_G), u_G.shape)
print(np.min(v_G), np.max(v_G), v_G.shape)
print('')


#=============== ST initial ======================================
ampl_C_initial = uv_plane[5][1:1024,0:513]
print(ampl_C_initial[0,0])
print(ampl_C_initial[511,0])
print(ampl_C_initial[1022,0])
print(ampl_C_initial.shape)
u,v = subs.make_axis_lists_2D(hdr2)
u_C = np.flip(u[1:514],axis=0)
v_C = v[1:1024]
print(np.min(u_C), np.max(u_C), u_C.shape)
print(np.min(v_C), np.max(v_C), v_C.shape)
print('')

#=============== ST feathered ======================================
ampl_C_feath = uv_plane[6][1:1024,0:513]
print(ampl_C_feath[0,0])
print(ampl_C_feath[511,0])
print(ampl_C_feath[1022,0])
print(ampl_C_feath.shape)
u,v = subs.make_axis_lists_2D(hdr2)
u_C = np.flip(u[1:514],axis=0)
v_C = v[1:1024]
print(np.min(u_C), np.max(u_C), u_C.shape)
print(np.min(v_C), np.max(v_C), v_C.shape)
print('')

radius_C = np.zeros([1023,513])
radius_G = np.zeros([1023,513])

for i in range(0,513):
    for j in range(0,1023):
        #radius_GC[j,i] = np.sqrt((u_GC[i])**2+(v_GC[j])**2)
        radius_G[j,i] = np.sqrt((u_G[i])**2+(v_G[j])**2)
        radius_C[j,i] = np.sqrt((u_C[i])**2+(v_C[j])**2)


In [ ]:
rad_samples = np.linspace(0,90,31)
rad_samples_centre = np.linspace(1.5,88.5,30)
print(rad_samples)
print(rad_samples_centre)
print('')
SA_orig_samples = np.zeros(30)
SA_match_samples = np.zeros(30)
SA_feath_samples = np.zeros(30)
ST_orig_samples = np.zeros(30)
ST_feath_samples = np.zeros(30)

for i in range(0,np.size(rad_samples)-1):
    wx = np.where((radius_C>=rad_samples[i]) & (radius_C<rad_samples[i+1]))[1]
    wy = np.where((radius_C>=rad_samples[i]) & (radius_C<rad_samples[i+1]))[0]
    SA_orig_samples[i] = np.mean(ampl_G_initial[wy,wx])
    SA_match_samples[i] = np.mean(ampl_G_match[wy,wx])
    SA_feath_samples[i] = np.mean(ampl_G_feath[wy,wx])
    ST_orig_samples[i] = np.mean(ampl_C_initial[wy,wx])
    ST_feath_samples[i] = np.mean(ampl_C_feath[wy,wx])

In [ ]:
fs=22
lw = 2

axs = ['ax1','ax2','ax3']

fig = plt.figure(figsize=(17,15))
plt.subplots_adjust(top = 0.98, bottom = 0.06, right = 0.98, left = 0.08, hspace=0.1)

axs[0] = fig.add_subplot(311)
axs[0].scatter(radius_C,ampl_C_initial,s=60,label='ST: original visibilities')
axs[0].scatter(radius_G,ampl_G_initial,s=50,label='SA: visibilities after initial taper')
axs[0].plot(rad_samples_centre,ST_orig_samples,color='blue',linewidth=lw,label='ST: mean')
axs[0].plot(rad_samples_centre,SA_orig_samples,color='red',linewidth=lw,label='SA: mean')
handles,labels = axs[0].get_legend_handles_labels()
order = [2,0,3,1]
axs[0].set_ylim(0,4000)
axs[0].legend([handles[idx] for idx in order], [labels[idx] for idx in order],fontsize=fs)

axs[1] = fig.add_subplot(312)
axs[1].scatter(radius_C,ampl_C_initial,s=60,label='ST: original visibilities')
axs[1].scatter(radius_G,ampl_G_match,s=50,label='SA: visibilities matched to ST beam')
axs[1].plot(rad_samples_centre,ST_orig_samples,color='blue',linewidth=lw,label='ST: mean')
axs[1].plot(rad_samples_centre,SA_match_samples,color='red',linewidth=lw,label='SA: mean')
axs[1].set_ylim(0,1800)
handles,labels = axs[1].get_legend_handles_labels()
order = [2,0,3,1]
axs[1].legend([handles[idx] for idx in order], [labels[idx] for idx in order],fontsize=fs)

axs[2] = fig.add_subplot(313)
axs[2].set_xlabel('$\sqrt{u^2+v^2}$ (m)',fontsize=fs)
axs[2].scatter(radius_C,ampl_C_feath,s=60,label='ST: visibilities feathered')
axs[2].scatter(radius_G,ampl_G_feath,s=50,label='SA: visibilities feathered')
axs[2].plot(rad_samples_centre,ST_feath_samples,color='blue',linewidth=lw,label='ST: mean')
axs[2].plot(rad_samples_centre,SA_feath_samples,color='red',linewidth=lw,label='SA: mean')
axs[2].set_ylim(0,1800)
handles,labels = axs[2].get_legend_handles_labels()
order = [2,0,3,1]
axs[2].legend([handles[idx] for idx in order], [labels[idx] for idx in order],fontsize=fs)

for i in range(0,3):
    axs[i].tick_params(axis="x", labelsize=fs)
    axs[i].tick_params(axis="y", labelsize=fs)
    axs[i].set_xlim(0,50)
    axs[i].set_ylabel('PI (K - uv-plane equiv.)',fontsize=fs)
    axs[i].plot([12.858,12.858],[0,80000],color='black',linestyle='dashed')
    axs[i].plot([17.144,17.144,],[0,80000],color='black',linestyle='dashed')
    axs[i].set_xticks([0,5,10,15,20,25,30,35,40,45,50])
    #axs[i].set_ticklabels(fontsize=fs)

plt.savefig('FIGURES/combining_radial_plots.pdf')

In [ ]:
w = np.where((radius_C > 12.858) & (radius_C < 17.144))

plt.scatter(ampl_C_initial[w],ampl_G_initial[w])
plt.xlim(0,700)
plt.ylim(0,700)
plt.axes().set_aspect('equal')

In [ ]:
w = np.where((radius_C > 12.858) & (radius_C < 17.144))

plt.scatter(ampl_C_initial[w],ampl_G_match[w])
plt.xlim(0,700)
plt.ylim(0,700)
plt.axes().set_aspect('equal')